In [30]:
import os 
import numpy as np
import pandas as pd

#
import re
import string
from nltk.corpus import stopwords

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# feature extractors 
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.metrics import classification_report, confusion_matrix

In [2]:
news_data_path = r'D:\new_ai\NLP_basic_to_advanced\bbc-news_text.csv'
news_data= pd.read_csv(news_data_path)

In [3]:
print(news_data.head())
print(f" {'-'*30} dataset {'-'*30}")
print(news_data.describe())
print(news_data['category'].value_counts())
print(f"  {'-'*30} Missing values {'-'*30}   ")
print(news_data.isnull().sum())
print(f"  {'-'*30} Duplicate values {'-'*30}   ")

print(news_data.duplicated().value_counts())

        category                                               text
0           tech  tv future in the hands of viewers with home th...
1       business  worldcom boss  left books alone  former worldc...
2          sport  tigers wary of farrell  gamble  leicester say ...
3          sport  yeading face newcastle in fa cup premiership s...
4  entertainment  ocean s twelve raids box office ocean s twelve...
 ------------------------------ dataset ------------------------------
       category                                               text
count      2225                                               2225
unique        5                                               2126
top       sport  ocean s twelve raids box office ocean s twelve...
freq        511                                                  2
category
sport            511
business         510
politics         417
tech             401
entertainment    386
Name: count, dtype: int64
  ------------------------------ Missing value

In [ ]:
# dup_rows = news_data[news_data.duplicated(subset='text', keep=False)]
# dup_rows.sort_values('text')


# dup_cols = news_data[news_data.duplicated(subset='category', keep=False)]
# dup_cols.sort_values('category')


,category,text
1484,business,tsunami to cost sri lanka $1.3bn sri lanka fac...
1251,business,china bans new tobacco factories the world s b...
1250,business,turkey-iran mobile deal at risk turkey s inv...
540,business,bank holds interest rate at 4.75% the bank of ...
543,business,economy strong in election year uk businesse...
...,...,...
637,tech,nintendo ds makes its euro debut nintendo s ds...
1181,tech,apple unveils low-cost mac mini apple has un...
644,tech,rings of steel combat net attacks gambling is ...
619,tech,virgin radio offers 3g broadcast uk broadcaste...


In [4]:
news_data.drop_duplicates(inplace=True)
print(news_data.head())

        category                                               text
0           tech  tv future in the hands of viewers with home th...
1       business  worldcom boss  left books alone  former worldc...
2          sport  tigers wary of farrell  gamble  leicester say ...
3          sport  yeading face newcastle in fa cup premiership s...
4  entertainment  ocean s twelve raids box office ocean s twelve...


In [5]:
# complete pre processing html remove 
punc = string.punctuation
sw_list = stopwords.words('english')
def remove_html(input_text):
   htm_removed= re.sub(re.compile('<.*?>'),'', input_text)
   return htm_removed

def punctuatio_remove(input_text):
   table = str.maketrans('','',punc) 
   removed_punc= input_text.translate(table)
   return removed_punc

def remove_stopwords(input_text):
   words= input_text.split()
   filtered_words=[]
   for word in words:
      if word.lower() not in sw_list:
         filtered_words.append(word)
   return " ".join(filtered_words)
   

In [6]:
# apply pre-processing on evrythning
news_data['text']=news_data['text'].apply(remove_html)
news_data['text']=news_data['text'].apply(remove_stopwords)
news_data['text']=news_data['text'].apply(punctuatio_remove)
print("Data Pre-processed ")
print(news_data.describe())

Data Pre-processed 
       category                                               text
count      2126                                               2126
unique        5                                               2120
top       sport  moya emotional davis cup win carlos moya descr...
freq        504                                                  2


In [7]:
X= news_data['text']
y=news_data['category']

le = LabelEncoder()
y= le.fit_transform(y)
print(y)

[4 0 3 ... 1 2 3]


In [23]:
X_train,X_test,y_train, y_test = train_test_split(X,y,test_size=0.3, random_state=42, stratify=y)  # all classes in train test in same proportion using stratify=y
print(f"shape of X Train is {X_train.shape}\n")
print(f"shape of y Train is {y_train.shape}\n")

#CountVectorizer, TfidfVectorizer
# Count vectorizer is BOw I love AI - 'I', 'Love', 'AI'
#TfidfVectorizer uses bow counts 


bow_cv_feat =CountVectorizer()
X_train_bow  = bow_cv_feat.fit_transform(X_train)
#print(bow_cv_feat.get_feature_names_out())
print("Vocabulary Size (BoW):", len(bow_cv_feat.get_feature_names_out()))
print("First 20 words:", bow_cv_feat.get_feature_names_out()[:20])
print('Count Vectorizer Done')

tfidf_feat =TfidfVectorizer()
X_train_tfidf  = tfidf_feat.fit_transform(X_train)
#print(tfidf_feat.get_feature_names_out())
print("Vocabulary Size (TF-IDF):", len(tfidf_feat.get_feature_names_out()))
print("First 20 words:", tfidf_feat.get_feature_names_out()[:20])
print('TF IDf Feature Done')

# transform the test data using same feature extractor

X_test_bow = bow_cv_feat.transform(X_test)
X_test_tfid = tfidf_feat.transform(X_test)


shape of X Train is (1488,)

shape of y Train is (1488,)

Vocabulary Size (BoW): 27968
First 20 words: ['00' '000' '0001' '000300' '00051' '000ayear' '000m' '000s' '000seat'
 '000seater' '000strong' '000th' '000vote' '001' '001and' '001st' '002'
 '003' '01' '0100']
Count Vectorizer Done
Vocabulary Size (TF-IDF): 27968
First 20 words: ['00' '000' '0001' '000300' '00051' '000ayear' '000m' '000s' '000seat'
 '000seater' '000strong' '000th' '000vote' '001' '001and' '001st' '002'
 '003' '01' '0100']
TF IDf Feature Done


# Multinomia NB 
It is a supervised ML appraoch mainly used for count-based discrete data and widely used for text classification such as spam, text, mail . Like it counts how many time a word appears in email. "buy"-> promotional or spam

In [50]:
#classifiers 
from sklearn.naive_bayes import MultinomialNB
mnb_clf = MultinomialNB()
mnb_model = mnb_clf.fit(X_train_bow,y_train)
multinomial_preds= mnb_clf.predict(X_test_bow)
#classification_report, confusion_matrix
class_report = classification_report(y_true=y_test,y_pred=multinomial_preds)
print(f" {'--'*20} Classification Report {'--'*20}\n")
print(class_report)
print(f" {'--'*20} Confusion Matrix {'--'*20}\n")
print(confusion_matrix(y_true=y_test,y_pred=multinomial_preds))

# accuracy score 
train_accuracy = mnb_clf.score(X_train_bow,y_train)
test_accuracy =mnb_clf.score(X_test_bow,y_test)

print(f" {'--'*20} Results {'--'*20}\n")
print(f"Training and Testing Performance Multinomial NB\n")
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy:  {test_accuracy:.4f}")

 ---------------------------------------- Classification Report ----------------------------------------

              precision    recall  f1-score   support

           0       0.97      0.97      0.97       151
           1       0.98      0.95      0.96       111
           2       0.95      0.98      0.97       121
           3       1.00      0.99      1.00       151
           4       0.95      0.96      0.96       104

    accuracy                           0.97       638
   macro avg       0.97      0.97      0.97       638
weighted avg       0.97      0.97      0.97       638

 ---------------------------------------- Confusion Matrix ----------------------------------------

[[146   1   1   0   3]
 [  1 105   3   0   2]
 [  2   0 119   0   0]
 [  0   0   1 150   0]
 [  2   1   1   0 100]]
 ---------------------------------------- Results ----------------------------------------

Training and Testing Performance Multinomial NB

Training Accuracy: 0.9933
Testing Accuracy:  0.

In [34]:
from sklearn.metrics import classification_report

# Training predictions
train_preds = mnb_clf.predict(X_train_bow)
print("--- TRAINING REPORT ---")
print(classification_report(y_train, train_preds))

# Testing predictions (What you already did)
print("--- TESTING REPORT ---")
print(classification_report(y_test, multinomial_preds))


--- TRAINING REPORT ---
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       352
           1       1.00      1.00      1.00       258
           2       0.99      0.99      0.99       282
           3       1.00      1.00      1.00       353
           4       0.98      1.00      0.99       243

    accuracy                           0.99      1488
   macro avg       0.99      0.99      0.99      1488
weighted avg       0.99      0.99      0.99      1488

--- TESTING REPORT ---
              precision    recall  f1-score   support

           0       0.97      0.97      0.97       151
           1       0.98      0.95      0.96       111
           2       0.95      0.98      0.97       121
           3       1.00      0.99      1.00       151
           4       0.95      0.96      0.96       104

    accuracy                           0.97       638
   macro avg       0.97      0.97      0.97       638
weighted avg       0.97      0

# Logistic Regression 


In [51]:
#classifiers 
from sklearn.linear_model import LogisticRegression
lr_clf = LogisticRegression()
lr_model = lr_clf.fit(X_train_bow,y_train)
lr_preds= lr_model.predict(X_test_bow)
#classification_report, confusion_matrix
class_report = classification_report(y_true=y_test,y_pred=lr_preds)
print(f" {'--'*20} Classification Report {'--'*20}\n")
print(class_report)
print(f" {'--'*20} Confusion Matrix {'--'*20}\n")
print(confusion_matrix(y_true=y_test,y_pred=lr_preds))

# accuracy score 
train_accuracy = lr_clf.score(X_train_bow,y_train)
test_accuracy =lr_clf.score(X_test_bow,y_test)

print(f" {'--'*20} Results {'--'*20}\n")
print(f"Training and Testing Performance LR\n")
print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Testing Accuracy:  {test_accuracy:.4f}")

 ---------------------------------------- Classification Report ----------------------------------------

              precision    recall  f1-score   support

           0       0.95      0.97      0.96       151
           1       0.99      0.96      0.98       111
           2       0.98      0.97      0.97       121
           3       0.97      1.00      0.99       151
           4       0.98      0.97      0.98       104

    accuracy                           0.97       638
   macro avg       0.98      0.97      0.98       638
weighted avg       0.98      0.97      0.97       638

 ---------------------------------------- Confusion Matrix ----------------------------------------

[[146   1   2   1   1]
 [  1 107   0   2   1]
 [  3   0 117   1   0]
 [  0   0   0 151   0]
 [  3   0   0   0 101]]
 ---------------------------------------- Results ----------------------------------------

Training and Testing Performance LR

Training Accuracy: 1.0000
Testing Accuracy:  0.9749
